# EDA — Phase-transition video curation

Lance-native exploration of `videos_raw` for the videogen pipeline.

Workflow:
1. Distributions over the source corpus (`duration_s`, `fps`, `caption_length`).
2. Tier-1 keyword counts (does the corpus actually contain phase-transition clips?).
3. Full-text search the caption column (FTS).
4. (Tier 2, deferred until GPU UDFs land) CLIP text→video retrieval, motion-strength filter, MTScore filter.
5. Sketch the final `phase_transitions_curated_train` view.

Cells below assume `data/videos/lancedb/videos_raw.lance` exists. Run from the example root:

```
python -m videogen.ingest_chronomagic --synthetic 1000 --overwrite
python -m videogen.backfill_geneva --tier 1
```

In [ ]:
import lancedb, pyarrow as pa, pandas as pd
from videogen.spec_queries import (
    summarise, distribution, preview, fts, ensure_caption_fts,
    keyword_spec, union_keyword_spec, curated_spec,
)

DB = 'data/videos/lancedb'
tbl = lancedb.connect(DB).open_table('videos_raw')
print(f'rows={len(tbl):,}  version={tbl.version}')

## 1 · Tier-1 summary

In [ ]:
pd.DataFrame(summarise(tbl), columns=['slice', 'rows'])

## 2 · Distributions

In [ ]:
df = tbl.search().select(['duration_s', 'fps', 'n_frames']).to_pandas()
df.describe()

In [ ]:
# caption length histogram — useful for finding overly short or template-y captions
lens = tbl.search().select(['caption_length']).to_pandas()['caption_length']
lens.hist(bins=40)

## 3 · Per-transition rates

In [ ]:
from videogen.schema import PHASE_TRANSITIONS
rows = []
for t in PHASE_TRANSITIONS:
    rows.append((t,
                  tbl.count_rows(filter=keyword_spec(t, 'train')),
                  tbl.count_rows(filter=keyword_spec(t, 'val'))))
pd.DataFrame(rows, columns=['transition', 'train', 'val'])

In [ ]:
# Sample 5 melting captions
preview(tbl, keyword_spec('melting'), n=5, columns=['clip_id', 'caption', 'duration_s', 'fps'])

## 4 · FTS — surface clips a keyword regex misses

Keyword regex on `caption` catches only the obvious forms (melt/melts/melting). The corpus contains many near-misses: "liquefies", "thaws", "vaporises". FTS picks them up without us extending the regex.

In [ ]:
ensure_caption_fts(tbl)
fts(tbl, 'ice cream', n=5)

## 5 · Tier-2 (deferred)

Once `backfill_geneva --tier 2` lands these cells become live:

```python
# CLIP text → video retrieval
import open_clip, torch
model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
with torch.no_grad():
    vec = model.encode_text(open_clip.tokenize(['ice melting into water']))[0]
    vec = (vec / vec.norm()).cpu().tolist()
tbl.search(vec, vector_column_name='clip_emb_video').metric('cosine').limit(8).to_pandas()

# Quality-gated count
tbl.count_rows(filter=curated_spec('train'))
```